# U information sharing: result explorer

This notebook reads the outputs from `analyze_u_information_sharing.py`. It has a whole-test-set view and a test-example view.

For any selected target concept C?, it places the two graphs side by side and lists every source group U?: whether Graph A (`replace_000`) and Graph B (`replace_100`) allow U??C?, how well a probe reads C? from U?, the extra predictive signal estimate, and the effect of removing U? for a selected test image. This helps inspect hypotheses such as ?C? is predicted from U? and U? in Graph A, while Graph B changes the allowed sources.?

The graph direction in this notebook is **U source group ? concept output**. `allowed` means the learned U2C mask permits that direct connection. The U-group ablation measures what changes in the actual masked model when one whole group is set to zero.

Set `CANDIDATE_IDS` and `SELECTED_CONCEPT` in the first code cell. The first two candidate images can be displayed using the experiment's test dataset loader.

> Rerun the analysis job after updating `analyze_u_information_sharing.py`. The current code saves per-test-row group-ablation results and patching records for all U-group/concept pairs.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RESULTS_DIR = PROJECT_ROOT / "experiments/u_information_sharing/celeba_fixed_backbone_u_ib_replace_000_vs_replace_100"

# Pick one or two row IDs from the saved CREAM test-set order.
CANDIDATE_IDS = [0, 1]
# None selects concept index 3 (High_Cheekbones in this CelebA experiment).
SELECTED_CONCEPT = None
# Optional alternative: SELECTED_CONCEPT = "Double_Chin"

print("Project:", PROJECT_ROOT)
print("Analysis results:", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())


## 1. What files are available?

The `.npz` activation files contain arrays for each split. The CSV files contain readable summaries. The test split used here is the same test split constructed from the experiment config.

If the output folder is missing, or if the patching outputs do not have `graph_allowed`, rerun the Condor analysis job after syncing the updated analysis script and cached-feature fix.


In [ ]:
expected = [
    "test_concept_accuracy_by_concept.csv",
    "single_test_group_ablation_long.csv",
    "probe_auc_long.csv",
    "conditional_information_long.csv",
    "within_model_activation_patch_pairs.csv",
    "within_model_activation_patch_summary.csv",
    "within_model_activation_patch_specificity.csv",
    "cross_model_activation_patch_summary.csv",
    "topology_masks_long.csv",
]
files = pd.DataFrame({"file": expected,
                      "present": [(RESULTS_DIR / f).is_file() for f in expected]})
display(files)


In [ ]:
def read_csv(name):
    path = RESULTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the analysis job and check its output_dir.")
    return pd.read_csv(path)

accuracy = read_csv("test_concept_accuracy_by_concept.csv")
ablation = read_csv("single_test_group_ablation_long.csv")
probe = read_csv("probe_auc_long.csv")
conditional = read_csv("conditional_information_long.csv")
patch_pairs = read_csv("within_model_activation_patch_pairs.csv")
patch_summary = read_csv("within_model_activation_patch_summary.csv")
specificity = read_csv("within_model_activation_patch_specificity.csv")
cross_patch = read_csv("cross_model_activation_patch_summary.csv")
topology = read_csv("topology_masks_long.csv")

models = accuracy["model"].drop_duplicates().tolist()
concepts = accuracy.sort_values("concept_index")["concept_name"].drop_duplicates().tolist()
print("Models:", models)
if SELECTED_CONCEPT is None:
    SELECTED_CONCEPT = concepts[min(3, len(concepts) - 1)]
print("Concept count:", len(concepts), "| selected concept:", SELECTED_CONCEPT)
print("Test rows per model:", read_csv("test_concept_predictions_long.csv").groupby("model")["image_id"].nunique().to_dict())


## 2. Whole-test-set concept prediction accuracy

This is the model's own concept prediction accuracy on the full test set. A concept score is thresholded at 0.5, then compared with the ground-truth concept label. It is not probe accuracy.


In [ ]:
display(accuracy.pivot(index="concept_name", columns="model", values="test_concept_accuracy").reindex(concepts).style.format("{:.3f}"))
accuracy.pivot(index="concept_name", columns="model", values="test_concept_accuracy").reindex(concepts).plot(
    kind="bar", figsize=(12, 4), ylim=(0, 1), ylabel="Test accuracy", title="CREAM concept accuracy by concept"
)
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()


## 3. Can a probe read concept information from each U group?

Each cell is test AUROC for a probe trained to predict a target concept from one source U group. This measures **decodability**: whether the signal can be read by this probe. It does not by itself show that CREAM's U2C layer uses that signal.


In [ ]:
group_probe = probe[probe["feature_set"].str.startswith("group_")].copy()
group_probe["source_group_index"] = group_probe["feature_set"].str.extract(r"group_(\d+)").astype(int)
for model in models:
    part = group_probe[group_probe.model == model]
    mat = part.pivot(index="target_concept", columns="source_group_index", values="test_auc").reindex(index=concepts)
    mat.columns = [f"U_{i}" for i in mat.columns]
    fig, ax = plt.subplots(figsize=(11, 6))
    im = ax.imshow(mat.values, vmin=0.5, vmax=1, cmap="viridis", aspect="auto")
    ax.set(title=f"{model}: concept predictability from each U group", xlabel="Source U group", ylabel="Target concept")
    ax.set_xticks(range(len(mat.columns)), mat.columns, rotation=45)
    ax.set_yticks(range(len(mat.index)), mat.index)
    fig.colorbar(im, ax=ax, label="Test AUROC")
    plt.tight_layout(); plt.show()


## 4. Does U? add information about C? beyond its own label C??

The saved proxy is `NLL(C_i | C_k) - NLL(C_i | C_k, U_k)`. Positive values mean adding U? improved held-out prediction of C? after the probe already knew the true C? label. Values are predictive log-loss improvements, not exact mutual information.

The diagonal U??C? is excluded by construction. Use the graph filter below to compare permitted and forbidden off-diagonal pairs.


In [ ]:
edge_flags = topology[["model", "source_group_index", "target_concept_index", "allowed"]].rename(
    columns={"target_concept_index": "target_index"}
)
conditional = conditional.merge(edge_flags, on=["model", "source_group_index", "target_index"], how="left")
conditional["edge_type"] = np.where(conditional.allowed.fillna(False), "graph allowed", "graph forbidden")
for model in models:
    part = conditional[conditional.model == model]
    mat = part.pivot(index="target_concept", columns="source_concept", values="conditional_info_proxy_nats").reindex(index=concepts, columns=concepts)
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(mat.values, cmap="coolwarm", aspect="auto")
    ax.set(title=f"{model}: extra predictive signal in U_k for C_i", xlabel="Source group U_k", ylabel="Target concept C_i")
    ax.set_xticks(range(len(concepts)), concepts, rotation=55, ha="right")
    ax.set_yticks(range(len(concepts)), concepts)
    fig.colorbar(im, ax=ax, label="NLL improvement (nats)")
    plt.tight_layout(); plt.show()

print("Mean proxy by graph edge status (off diagonal):")
display(conditional.groupby(["model", "edge_type"], as_index=False).conditional_info_proxy_nats.agg(["mean", "median", "count"]))

# One concept-focused table: each row is a possible source U group for C_i.
focus = SELECTED_CONCEPT
connectivity = topology[topology.target_concept == focus].pivot(
    index="source_group_index", columns="model", values="allowed"
)
probe_for_focus = group_probe[group_probe.target_concept == focus].pivot(
    index="source_group_index", columns="model", values="test_auc"
)
info_for_focus = conditional[conditional.target_concept == focus].pivot(
    index="source_group_index", columns="model", values="conditional_info_proxy_nats"
)
image_for_focus = ablation[
    (ablation.image_id == CANDIDATE_IDS[0]) & (ablation.target_concept == focus)
].pivot(index="source_group_index", columns="model", values="signed_logit_change")

focus_table = pd.DataFrame({
    "U source group": [f"U{i} ({concepts[i]})" for i in range(len(concepts))],
    "Graph A allows U_k ? C_i": connectivity.reindex(range(len(concepts)))[models[0]].values,
    "Graph B allows U_k ? C_i": connectivity.reindex(range(len(concepts)))[models[1]].values,
})
for model, graph_name in zip(models, ["Graph A", "Graph B"]):
    focus_table[f"{graph_name}: probe AUROC"] = probe_for_focus.reindex(range(len(concepts)))[model].values
    focus_table[f"{graph_name}: extra signal (nats)"] = info_for_focus.reindex(range(len(concepts)))[model].values
    focus_table[f"{graph_name}: candidate logit change"] = image_for_focus.reindex(range(len(concepts)))[model].values
print(f"For C_i = {focus}; candidate row = {CANDIDATE_IDS[0]} (ablation change is ablated minus original logit).")
display(focus_table)


## 5. What changes when a U group is patched?

For matched test-image pairs, the analysis replaces one entire U group in a clean example with that group's activation from a source example. It records the change in each concept logit. `target_change` pairs differ in the named concept; `control_same_target` pairs keep that concept the same.

The output now includes every U??C? pair, including pairs forbidden by the graph. `graph_allowed` labels the trained mask. Effects are computed by passing the patched representation through the model's U2C layer, so a forbidden connection remains masked during this calculation; for a direct one-layer mask it should have no effect. The forbidden rows are included so you can verify and compare them.


In [ ]:
if "graph_allowed" not in patch_pairs.columns:
    print("These patching outputs are from an older run and omit forbidden graph pairs. Rerun the updated analysis job.")
else:
    print("Pair-level records:", len(patch_pairs))
    patch_summary["edge_type"] = np.where(patch_summary.graph_allowed, "allowed", "forbidden")
    specificity_edges = topology[["model", "source_concept", "target_concept", "allowed"]]
    specificity = specificity.merge(
        specificity_edges,
        left_on=["model", "source_group_concept", "predicted_concept"],
        right_on=["model", "source_concept", "target_concept"], how="left"
    )
    specificity["edge_type"] = specificity["allowed"].map({True:"allowed", False:"forbidden"})
    display(patch_summary.groupby(["model", "edge_type", "pair_type"], as_index=False).agg(
        mean_abs_logit_change=("mean_absolute_logit_change", "mean"),
        mean_signed_logit_change=("mean_signed_logit_change", "mean"),
        pathway_count=("predicted_concept", "count")
    ))

    # Q compares response to target-changing pairs against matched controls.
    # Positive Q means the target-changing swap had the larger absolute logit effect.
    if "specificity_Q_target_minus_control" in specificity:
        display(specificity.groupby(["model", "edge_type"], as_index=False).agg(
            mean_Q=("specificity_Q_target_minus_control", "mean"),
            median_Q=("specificity_Q_target_minus_control", "median"),
            positive_paths=("specificity_Q_target_minus_control", lambda x: int((x > 0).sum())),
            paths=("specificity_Q_target_minus_control", "count")
        ))


### Choose a target concept and read its incoming graph connections

The table above answers: for the selected C?, which U groups are graph-allowed in Graph A and Graph B? It also lines those connections up with probe AUROC, conditional predictive information, and the selected test row's U-group ablation effect. The diagonal conditional-information value is blank because that estimate intentionally excludes C? conditioned on itself.

You can change `SELECTED_CONCEPT` in the first cell to examine another concept. The example image's actual U?C effect is the `candidate logit change` column; the graph-allowed columns tell you which direct inputs can contribute under each model's mask.


In [ ]:
SHOW_ONLY_GRAPH_ALLOWED = False
EXCLUDE_ASSIGNED_GROUP = True

if "graph_allowed" in patch_summary.columns:
    edge_table = patch_summary.copy()
    if SHOW_ONLY_GRAPH_ALLOWED:
        edge_table = edge_table[edge_table.graph_allowed]
    if EXCLUDE_ASSIGNED_GROUP:
        edge_table = edge_table[edge_table.source_group_concept != edge_table.predicted_concept]
    edge_table = edge_table.sort_values("mean_absolute_logit_change", ascending=False)
    display(edge_table[["model", "source_group_concept", "predicted_concept", "graph_allowed", "changed_concept", "pair_type", "n_pairs", "mean_signed_logit_change", "mean_absolute_logit_change"]].head(40))


## 6. Compare the two graph variants on the same test images

Cross-model transplantation inserts a donor model's U group into the receiver model's U vector for the same test image, then measures the receiver's concept-logit change. Raw and validation-aligned transplants are different diagnostics: alignment reduces coordinate mismatch, but it is fitted on validation data and does not prove that the two models learned the same representation.


In [ ]:
display(cross_patch.sort_values(["receiver", "mode", "mean_absolute_logit_change"], ascending=[True, True, False]).head(40))


## 7. Inspect one or two test examples

`CANDIDATE_IDS` are row numbers in the analysis test-loader order. For each candidate, the notebook shows every concept's ground truth, score, logit, predicted label, and correctness for both graphs. It also tries to display the source image from the same configured CelebA test split.

For the selected concept, the per-image table shows the change after zeroing each U group, separately for Graph A and Graph B. Set `CANDIDATE_EFFECT_GRAPH_ALLOWED_ONLY = True` to show only graph-allowed U groups. Set `CANDIDATE_EFFECT_EXCLUDE_ASSIGNED_GROUP = False` to include U?.


In [ ]:
predictions = read_csv("test_concept_predictions_long.csv")
selected = predictions[predictions.image_id.isin(CANDIDATE_IDS)].copy()
selected["predicted_label"] = selected.concept_score >= 0.5
selected["true_label"] = selected.concept_target >= 0.5
selected["correct"] = selected.predicted_label == selected.true_label

for image_id in CANDIDATE_IDS:
    print(f"Test row {image_id}: all concepts, with Graph A and Graph B side by side")
    one = selected[selected.image_id == image_id]
    table = one.pivot(index=["concept_index", "concept_name"], columns="model",
                      values=["true_label", "concept_score", "concept_logit", "predicted_label", "correct"])
    display(table.reindex(index=sorted(table.index)))

# Show the actual test image and its annotated concepts, when the dataset is available locally.
try:
    import yaml
    import torch
    from src.utils import get_component_with_dicts
    config_path = PROJECT_ROOT / "all_configs/fixed_backbone/celeba_concept_concept_fixcount_topology_no_side_channel_fixed_backbone/CREAM_celeba_concept_concept_fixcount_topology_no_side_channel_fixed_backbone_only_replace_000.yaml"
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    dataset_class = get_component_with_dicts("dataset", config["dataset_name"])
    dataset = dataset_class(**config.get("dataset_params", {}))
    dataset.setup("test")
    test_set = dataset.test_data
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    for image_id in CANDIDATE_IDS:
        image, true_concepts, _ = test_set[image_id]
        image = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(image); ax.axis("off")
        labels = [f"{concepts[i]}: {bool(float(v) > .5)}" for i, v in enumerate(true_concepts)]
        ax.set_title(f"Test row {image_id}\n" + " | ".join(labels), fontsize=9)
        plt.tight_layout(); plt.show()
except Exception as exc:
    print("Could not display the source image. Check dataset files/config on this machine.")
    print(type(exc).__name__ + ":", exc)

# For the selected C_i, each row shows what happens if a single U_k is zeroed.
CANDIDATE_EFFECT_GRAPH_ALLOWED_ONLY = False
CANDIDATE_EFFECT_EXCLUDE_ASSIGNED_GROUP = True
candidate_effects = ablation[ablation.image_id.isin(CANDIDATE_IDS)].copy()
if CANDIDATE_EFFECT_GRAPH_ALLOWED_ONLY:
    candidate_effects = candidate_effects[candidate_effects.graph_allowed]
if CANDIDATE_EFFECT_EXCLUDE_ASSIGNED_GROUP:
    candidate_effects = candidate_effects[candidate_effects.source_group_index != candidate_effects.target_concept_index]
for image_id in CANDIDATE_IDS:
    for model in models:
        part = candidate_effects[(candidate_effects.image_id == image_id) &
                                 (candidate_effects.target_concept == SELECTED_CONCEPT) &
                                 (candidate_effects.model == model)]
        print(f"{model}, row {image_id}, target {SELECTED_CONCEPT}: each U group's effect")
        display(part.sort_values("source_group_index")[[
            "source_group_concept", "target_concept", "graph_allowed", "target_label",
            "baseline_logit", "ablated_logit", "signed_logit_change", "absolute_logit_change",
            "baseline_score", "ablated_score", "signed_score_change"
        ]])

# U representation for each candidate and each graph variant.
for model in models:
    npz_path = RESULTS_DIR / f"{model}_test_activations.npz"
    if not npz_path.exists():
        print("Missing", npz_path); continue
    arr = np.load(npz_path)
    for image_id in CANDIDATE_IDS:
        if image_id >= len(arr["u_concept_groups"]):
            print(f"Row {image_id} outside test split for {model}"); continue
        groups = arr["u_concept_groups"][image_id]
        fig, ax = plt.subplots(figsize=(11, 4))
        im = ax.imshow(groups, aspect="auto", cmap="magma")
        ax.set(title=f"{model}, test row {image_id}: U representation (group ? feature)", xlabel="Feature within group", ylabel="U group")
        ax.set_yticks(range(len(concepts)), [f"U{i}: {c}" for i, c in enumerate(concepts)])
        fig.colorbar(im, ax=ax, label="U activation")
        plt.tight_layout(); plt.show()


### Matched-pair patch records involving the selected examples

The table below is a second, different intervention: it replaces a U group with that group's activation from a matched source image. These records exist only when a chosen row was selected as either the clean or source member of one of the matched pairs. If no rows appear, the zero-group ablation table above still shows every group for the selected row.


In [ ]:
candidate_patch = patch_pairs[
    patch_pairs.clean_image_id.isin(CANDIDATE_IDS) | patch_pairs.source_image_id.isin(CANDIDATE_IDS)
].copy()
if candidate_patch.empty:
    print("No matched-pair patching record used these row IDs. Try other values in CANDIDATE_IDS.")
else:
    display(candidate_patch.sort_values(["clean_image_id", "source_group_index", "predicted_concept_index"])[[
        "model", "clean_image_id", "source_image_id", "pair_type", "changed_concept",
        "source_group_concept", "predicted_concept", "graph_allowed",
        "baseline_logit", "patched_logit", "signed_logit_change", "absolute_logit_change"
    ]])


## Reading the evidence carefully

- **Concept accuracy** describes the model's output performance.
- **Probe AUROC** describes whether a separate trained probe can decode a concept from U.
- **Conditional NLL improvement** asks whether U? adds predictive signal beyond the known C? label; it is a proxy, not exact mutual information.
- **Activation patching** measures how the U2C output changes after swapping an entire U group. It does not isolate individual neurons or prove semantic causation.
- **Graph-forbidden effects** are measured by the dense raw U2C function in an offline counterfactual. The actual masked forward pass zeros those weights, so those paths do not contribute during normal prediction.

Use the full-test-set summaries for broad patterns and the candidate view to understand what those aggregate quantities look like for concrete rows.
